## 1.Inisialisasi Environment dan Konfigurasi Awal
Pada tahap ini dilakukan persiapan seluruh komponen yang dibutuhkan sebelum proses ekstraksi dokumen dimulai.

Kode berikut melakukan beberapa langkah penting:
- Mengimpor library yang digunakan pada proses ekstraksi, pemrosesan dokumen, embedding, dan vector database.
- Memuat variabel lingkungan dari file `.env`.
- Mengambil API Key Gemini untuk mengakses model Google Gemini.
- Melakukan validasi API Key agar proses dapat berjalan dengan aman.
- Menginisialisasi client Gemini yang akan digunakan untuk ekstraksi informasi dan menjawab pertanyaan.
- Menentukan lokasi dokumen PDF yang akan diproses.
- Mengambil lokasi instalasi Poppler yang diperlukan oleh `pdf2image` untuk mengonversi PDF menjadi gambar.

Setelah tahap ini selesai, seluruh konfigurasi yang dibutuhkan oleh pipeline Multimodal RAG telah siap digunakan.

In [7]:
import os
import time
import json
from dotenv import load_dotenv
from pdf2image import convert_from_path
from google import genai
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("API Key tidak ditemukan! Pastikan file .env sudah benar.")

client = genai.Client(api_key=api_key)
print("SDK Google GenAI berhasil diinisialisasi.")

pdf_path = "D:\Proyek Magang\RAG\Bagian-A-Multimodal-Retrieval-Augmented-Generation\data\Laporan Keuangan Bank Mandiri 2025.pdf" 
jalur_poppler = os.getenv("POPPLER_PATH")

<>:20: SyntaxWarning: invalid escape sequence '\P'
<>:20: SyntaxWarning: invalid escape sequence '\P'
C:\Users\satri\AppData\Local\Temp\ipykernel_25252\443459258.py:20: SyntaxWarning: invalid escape sequence '\P'
  pdf_path = "D:\Proyek Magang\RAG\Bagian-A-Multimodal-Retrieval-Augmented-Generation\data\Laporan Keuangan Bank Mandiri 2025.pdf"


SDK Google GenAI berhasil diinisialisasi.


## 2.Mengonversi Dokumen PDF
Model Gemini Vision menerima input berupa gambar. Oleh karena itu dokumen PDF perlu dikonversi terlebih dahulu menjadi kumpulan gambar. Pada tahap ini setiap halaman PDF akan diubah menjadi sebuah objek gambar yang nantinya digunakan untuk proses ekstraksi informasi.

In [8]:
print(f"Memulai konversi dokumen {pdf_path}...")
try:
    halaman_gambar = convert_from_path(
        pdf_path, 
        dpi=200,
        poppler_path=jalur_poppler
    )
    print(f"Berhasil mengonversi total {len(halaman_gambar)} halaman menjadi format gambar.")
except Exception as e:
    print(f"Gagal membaca PDF. Pastikan nama file dan path Poppler benar.\nError: {e}")
    exit()

Memulai konversi dokumen D:\Proyek Magang\RAG\Bagian-A-Multimodal-Retrieval-Augmented-Generation\data\Laporan Keuangan Bank Mandiri 2025.pdf...
Berhasil mengonversi total 9 halaman menjadi format gambar.


## 3.Menyiapkan Prompt Ekstraksi
Prompt berikut digunakan untuk mengarahkan Gemini agar mengekstrak seluruh informasi penting yang terdapat pada dokumen.

Informasi yang ingin diperoleh meliputi:
- Teks utama dokumen.
- Tabel beserta isinya.
- Grafik atau diagram dalam bentuk deskripsi.
- Informasi visual lain yang relevan.

Hasil ekstraksi akan digunakan sebagai sumber pengetahuan untuk sistem RAG.

In [9]:
ekstraksi_prompt = """
Kamu adalah AI Data Extractor yang sangat teliti. Tugasmu adalah mengekstrak seluruh informasi dari gambar halaman dokumen ini.
Ikuti aturan ketat berikut:
1. TEKS: Ekstrak semua teks naratif persis seperti aslinya.
2. TABEL: Jika terdapat tabel data, konversikan tabel tersebut menjadi format Markdown yang rapi tanpa menghilangkan satu baris/kolom pun.
3. GRAFIK/INFOGRAFIS: Jika terdapat grafik, chart, atau alur infografis, berikan penjelasan deskriptif yang mendetail tentang apa maksud gambar tersebut, termasuk angka, persentase, dan teks di dalamnya.
4. Jangan menambahkan informasi dari luar, hanya berdasarkan apa yang ada di gambar.
"""

## 4.Ekstraksi Informasi dari Setiap Halaman
Pada tahap ini setiap halaman dokumen diproses menggunakan Gemini Vision.

Untuk setiap halaman:
1. Gambar dikirim ke model Gemini.
2. Informasi pada halaman diekstrak.
3. Hasil ekstraksi disimpan bersama metadata halaman.

Metadata halaman penting untuk membantu pelacakan sumber informasi ketika sistem menjawab pertanyaan pengguna.

In [10]:
dokumen_rag = []
print("\n Memulai proses ekstraksi Multimodal dengan Gemini 3.1 Flash Lite...")

for i, halaman in enumerate(halaman_gambar):
    nomor_halaman = i + 1
    print(f"[{nomor_halaman}/{len(halaman_gambar)}] Membedah halaman {nomor_halaman}...", end=" ")
    
    try:
        response = client.models.generate_content(
            model='gemini-3.1-flash-lite',
            contents=[ekstraksi_prompt, halaman]
        )
        
        dokumen_rag.append({
            "metadata": {"page": nomor_halaman, "source": "Laporan Bank Mandiri 2025"},
            "content": response.text
        })
        print("Berhasil!")
        
    except Exception as e:
        print(f"Gagal! (Error: {e})")
    
    if nomor_halaman < len(halaman_gambar):
        print(f"  Pendinginan API selama 15 detik untuk menghindari Rate Limit...")
        time.sleep(15)


 Memulai proses ekstraksi Multimodal dengan Gemini 3.1 Flash Lite...
[1/9] Membedah halaman 1... Berhasil!
  Pendinginan API selama 15 detik untuk menghindari Rate Limit...
[2/9] Membedah halaman 2... Berhasil!
  Pendinginan API selama 15 detik untuk menghindari Rate Limit...
[3/9] Membedah halaman 3... Berhasil!
  Pendinginan API selama 15 detik untuk menghindari Rate Limit...
[4/9] Membedah halaman 4... Berhasil!
  Pendinginan API selama 15 detik untuk menghindari Rate Limit...
[5/9] Membedah halaman 5... Berhasil!
  Pendinginan API selama 15 detik untuk menghindari Rate Limit...
[6/9] Membedah halaman 6... Berhasil!
  Pendinginan API selama 15 detik untuk menghindari Rate Limit...
[7/9] Membedah halaman 7... Berhasil!
  Pendinginan API selama 15 detik untuk menghindari Rate Limit...
[8/9] Membedah halaman 8... Berhasil!
  Pendinginan API selama 15 detik untuk menghindari Rate Limit...
[9/9] Membedah halaman 9... Berhasil!


## 5.Menyimpan Hasil Ekstraksi
Setelah seluruh halaman selesai diproses, hasil ekstraksi disimpan dalam format JSON. Penyimpanan ini bertujuan agar proses ekstraksi tidak perlu dijalankan ulang setiap kali notebook dieksekusi. File JSON yang dihasilkan akan menjadi knowledge base utama pada proses RAG.

In [11]:
output_file = r'D:\Proyek Magang\RAG\Bagian-A-Multimodal-Retrieval-Augmented-Generation\data\knowledge_base_mandiri.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(dokumen_rag, f, ensure_ascii=False, indent=4)

print(f"\n Seluruh hasil ekstraksi telah diamankan ke dalam file '{output_file}'.")


 Seluruh hasil ekstraksi telah diamankan ke dalam file 'D:\Proyek Magang\RAG\Bagian-A-Multimodal-Retrieval-Augmented-Generation\data\knowledge_base_mandiri.json'.


## 6.Memuat Knowledge Base dan Membuat LangChain Document
Pada tahap ini file JSON yang berisi hasil ekstraksi dokumen dibaca kembali ke dalam memori. Setiap data hasil ekstraksi kemudian dikonversi menjadi objek `Document` milik LangChain dengan memisahkan:

- `page_content` sebagai isi utama dokumen.
- `metadata` sebagai informasi tambahan seperti nomor halaman dan sumber dokumen.

Format `Document` digunakan sebagai standar input pada LangChain sehingga dokumen dapat diproses lebih lanjut pada tahap chunking, embedding, dan indexing.

In [12]:
with open(output_file, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

dokumen_langchain = []
for item in raw_data:
    doc = Document(
        page_content=item['content'],
        metadata=item['metadata']
    )
    dokumen_langchain.append(doc)

print(f"Berhasil memuat {len(dokumen_langchain)} halaman dari JSON.")

Berhasil memuat 9 halaman dari JSON.


## 7.Memecah Dokumen Menjadi Beberapa Chunk
Dokumen yang terlalu panjang kurang efektif untuk proses embedding dan retrieval. Oleh karena itu dokumen dibagi menjadi beberapa bagian yang lebih kecil menggunakan teknik chunking. Overlap digunakan agar informasi yang berada di batas antar chunk tidak hilang dan konteks tetap terjaga.

In [14]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000,
    chunk_overlap=500,
    separators=["\n\n", "\n", " ", ""] 
)

potongan_dokumen = text_splitter.split_documents(dokumen_langchain)
print(f"Dokumen berhasil dipotong menjadi {len(potongan_dokumen)} chunks ukuran besar.")

Dokumen berhasil dipotong menjadi 13 chunks ukuran besar.


## 8.Membuat Embedding dan Menyimpan Vector Database
Pada tahap ini setiap chunk dokumen diubah menjadi representasi numerik (embedding) menggunakan model `all-MiniLM-L6-v2`. Embedding memungkinkan sistem memahami hubungan makna antar teks sehingga pencarian dapat dilakukan berdasarkan kesamaan konteks, bukan hanya kecocokan kata. Setelah embedding dibuat, seluruh data disimpan ke dalam ChromaDB sebagai vector database. Database ini akan digunakan untuk melakukan pencarian semantik dan mengambil informasi yang paling relevan terhadap pertanyaan pengguna pada tahap Retrieval-Augmented Generation (RAG). Hasil dari proses ini adalah vector database yang tersimpan secara permanen dan siap digunakan untuk proses retrieval.

In [17]:
from pathlib import Path

print("Mengunduh/Memuat model Local Embedding (all-MiniLM-L6-v2)...")
model_embedding_lokal = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

BASE_DIR = Path(r"D:\Proyek Magang\RAG")
direktori_db = str(BASE_DIR / "mandiri_chroma_db")

print("Memasukkan data ke dalam ChromaDB...")
vector_db = Chroma.from_documents(
    documents=potongan_dokumen,
    embedding=model_embedding_lokal,
    persist_directory=direktori_db 
)

print(f"\n Vector Database telah tercipta dan tersimpan di folder '{direktori_db}'.")

Mengunduh/Memuat model Local Embedding (all-MiniLM-L6-v2)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4548.90it/s]


Memasukkan data ke dalam ChromaDB...

 Vector Database telah tercipta dan tersimpan di folder 'D:\Proyek Magang\RAG\mandiri_chroma_db'.


## 9.Memuat Vector Database
Pada tahap ini database vektor yang telah dibuat sebelumnya dimuat kembali ke memori. Langkah ini memungkinkan sistem melakukan pencarian tanpa perlu membuat embedding ulang untuk seluruh dokumen.

In [18]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

model_embedding_lokal = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

BASE_DIR = os.getcwd()
direktori_db = os.path.join(BASE_DIR, "mandiri_chroma_db")
vector_db = Chroma(persist_directory=direktori_db, embedding_function=model_embedding_lokal)
print("Database ChromaDB berhasil diakses!\n")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10581.04it/s]


Database ChromaDB berhasil diakses!



C:\Users\satri\AppData\Local\Temp\ipykernel_25252\3310182731.py:9: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(persist_directory=direktori_db, embedding_function=model_embedding_lokal)


## 10.Membangun Pipeline Retrieval-Augmented Generation (RAG)
Pipeline RAG menggabungkan kemampuan pencarian informasi dengan kemampuan generasi jawaban dari Large Language Model.

Alur yang dilakukan:
1. Pengguna mengajukan pertanyaan.
2. Sistem mencari chunk yang relevan pada ChromaDB.
3. Chunk tersebut digunakan sebagai konteks tambahan.
4. Gemini menghasilkan jawaban berdasarkan konteks yang ditemukan.

Pendekatan ini membantu menghasilkan jawaban yang lebih akurat dan sesuai dengan isi dokumen.

In [19]:
def tanya_laporan_mandiri(pertanyaan):
    print(f"Mencari jawaban untuk: '{pertanyaan}'\n")
    hasil_pencarian = vector_db.similarity_search(pertanyaan, k=15)
    konteks = ""
    
    for i, doc in enumerate(hasil_pencarian):
        halaman = doc.metadata.get('page', 'Tidak diketahui')
        konteks += f"---\n[SUMBER: HALAMAN {halaman}]\n{doc.page_content}\n"
        
    print("Menyerahkan jaring konteks ke Gemini untuk dianalisis...\n")
        
    prompt_rag = f"""
    Kamu adalah asisten AI yang cerdas dan ahli dalam menganalisis dokumen keuangan Bank Mandiri.
    Tugasmu adalah menjawab pertanyaan menggunakan HANYA informasi dari 'Konteks Dokumen' di bawah ini.
    Setiap potongan konteks diawali dengan tag [SUMBER: HALAMAN X].
    
    Aturan Wajib:
    1. Sajikan data angka nominal dan persentase dengan sangat akurat.
    2. Di akhir jawabanmu, buatlah satu baris khusus berbunyi: "Sumber Halaman: [Sebutkan nomor halamannya]". 
    3. HANYA sebutkan halaman tempat kamu benar-benar menemukan angka/jawaban tersebut. Jangan sebutkan halaman yang tidak relevan.

    Konteks Dokumen:
    {konteks}

    Pertanyaan: {pertanyaan}
    Jawaban:
    """
    
    response = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=prompt_rag
    )
    print(response.text)

## 11.Menguji Sistem RAG
Pada tahap terakhir dilakukan pengujian menggunakan beberapa pertanyaan terkait isi dokumen.

Tujuan pengujian adalah memastikan bahwa:
- Informasi dapat ditemukan melalui proses retrieval.
- Jawaban yang dihasilkan sesuai dengan dokumen.
- Sistem mampu memanfaatkan konteks yang relevan saat menjawab pertanyaan.

In [20]:
pertanyaan_1 = "Berdasarkan laporan, berapa pertumbuhan nominal dan pertumbuhan persentase dari sektor Tambang dan Konstruksi pada tahun 2025 dibandingkan tahun 2024?"
tanya_laporan_mandiri(pertanyaan_1)

print("\n\n")

pertanyaan_2 = "Apa saja saluran pengaduan yang disediakan oleh Bank Mandiri?"
tanya_laporan_mandiri(pertanyaan_2)

Mencari jawaban untuk: 'Berdasarkan laporan, berapa pertumbuhan nominal dan pertumbuhan persentase dari sektor Tambang dan Konstruksi pada tahun 2025 dibandingkan tahun 2024?'

Menyerahkan jaring konteks ke Gemini untuk dianalisis...

Berdasarkan data "Kredit yang Diberikan dan Piutang/Pembiayaan Syariah Berdasarkan Sektor Ekonomi" pada dokumen, berikut adalah rincian pertumbuhan untuk sektor Tambang dan Konstruksi pada tahun 2025 dibandingkan tahun 2024:

*   **Sektor Tambang:**
    *   Pertumbuhan Nominal: Rp11.614.853 juta (Rp11,61 triliun).
    *   Pertumbuhan Persentase: 7,98%.
*   **Sektor Konstruksi:**
    *   Pertumbuhan Nominal: Rp8.264.848 juta (Rp8,26 triliun).
    *   Pertumbuhan Persentase: 8,27%.

Sumber Halaman: [HALAMAN 4]



Mencari jawaban untuk: 'Apa saja saluran pengaduan yang disediakan oleh Bank Mandiri?'

Menyerahkan jaring konteks ke Gemini untuk dianalisis...

Berdasarkan dokumen yang tersedia, Bank Mandiri menyediakan berbagai saluran pengaduan bagi nasabah ya